# 4. MOFA+ native paired reference

This notebook trains one native three-view MOFA+ model using the 58 cells with
paired RNA, total-protein and phosphopeptide measurements. It follows the
structure of the first-manuscript MOFA+ training notebook:

1. load the preprocessed matrices;
2. convert them to MOFA long format;
3. define one Gaussian likelihood per view;
4. train one model with five factors and `seed=1`;
5. save one HDF5 model.

Protein and phosphopeptide missing values remain missing. Control and treated
cells are kept in a single group so that treatment-associated variation is not
removed before model fitting.

## Environment

Use the same MOFA environment as the first manuscript where possible. GPU
training also requires a CuPy build compatible with the installed CUDA version.
For example, CUDA 12 normally uses `cupy-cuda12x`.

```python
%pip install "mofapy2==0.7.2" "pandas==2.3.2" "numpy==2.1.2" h5py
%pip install cupy-cuda12x
```

Restart the Jupyter kernel after installation.

In [1]:
from __future__ import annotations

import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import numpy as np
import pandas as pd
from mofapy2.run.entry_point import entry_point


def package_version(package_name: str) -> str:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "not installed"


print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("mofapy2:", package_version("mofapy2"))
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.10.18
Platform: Windows-10-10.0.26200-SP0
mofapy2: 0.7.2
pandas: 2.3.2
NumPy: 2.2.6


## 1. Configuration

Normally `PROJECT_DIR_OVERRIDE` can remain `None`. If Jupyter was started from
another directory, set it to the directory containing `processed_data`, for
example:

```python
PROJECT_DIR_OVERRIDE = Path(r"C:\Users\49152\Downloads\2nd_paper\Script\1_Preprocessing")
```

In [4]:
PROJECT_DIR_OVERRIDE: Path | None = None

N_FACTORS = 5
SEED = 1
GPU_MODE = True
GPU_DEVICE = 0
OVERWRITE_MODEL = False


def find_project_dir(override: Path | None = None) -> Path:
    if override is not None:
        project_dir = Path(override).expanduser().resolve()
        if not (project_dir / "processed_data" / "mofa_input").is_dir():
            raise FileNotFoundError("PROJECT_DIR_OVERRIDE does not contain processed_data/mofa_input: " f"{project_dir}")
        return project_dir

    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "processed_data" / "mofa_input").is_dir():
            return candidate

    raise FileNotFoundError("Could not locate processed_data/mofa_input. " "Set PROJECT_DIR_OVERRIDE explicitly.")


PROJECT_DIR = find_project_dir(PROJECT_DIR_OVERRIDE)
PROCESSED_DIR = PROJECT_DIR / "processed_data"
INPUT_DIR = PROCESSED_DIR / "mofa_input"
OUTPUT_DIR = PROCESSED_DIR / "mofa_models" / "native_paired58"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_FILE = PROCESSED_DIR / "metadata_master_samples.csv"
RNA_FILE = INPUT_DIR / "mofa_rna_paired58_z.csv"
PROTEIN_FILE = INPUT_DIR / "mofa_protein_paired58_z_with_na.csv"
PHOSPHOPEPTIDE_FILE = INPUT_DIR / "mofa_phosphopeptide_paired58_z_with_na.csv"
MODEL_FILE = OUTPUT_DIR / "native_paired58_model.hdf5"

for input_file in (METADATA_FILE, RNA_FILE, PROTEIN_FILE, PHOSPHOPEPTIDE_FILE):
    if not input_file.is_file():
        raise FileNotFoundError(f"Required input file does not exist: {input_file}")

if MODEL_FILE.exists() and not OVERWRITE_MODEL:
    raise FileExistsError(f"Model file already exists: {MODEL_FILE}\n" "Set OVERWRITE_MODEL=True only if you intentionally want to replace it.")

print("Project directory:", PROJECT_DIR)
print("Output model:", MODEL_FILE)

Project directory: C:\Users\49152\Downloads\2nd_paper\Script\1_Preprocessing
Output model: C:\Users\49152\Downloads\2nd_paper\Script\1_Preprocessing\processed_data\mofa_models\native_paired58\native_paired58_model.hdf5


## 2. Confirm GPU availability

This explicit check prevents `mofapy2` from silently switching to CPU if CuPy
or the CUDA runtime is unavailable.

In [6]:
if GPU_MODE:
    try:
        import cupy as cp
    except ImportError as exc:
        raise ImportError("GPU_MODE=True, but CuPy is not installed in this Jupyter kernel. "
                          "Install the CuPy package matching your CUDA version and restart the kernel.") from exc

    device_count = int(cp.cuda.runtime.getDeviceCount())
    if device_count == 0:
        raise RuntimeError("GPU_MODE=True, but CuPy detected no CUDA devices.")
    if GPU_DEVICE < 0 or GPU_DEVICE >= device_count:
        raise ValueError(f"GPU_DEVICE={GPU_DEVICE}, but CuPy detected {device_count} device(s).")

    cp.cuda.Device(GPU_DEVICE).use()
    gpu_properties = cp.cuda.runtime.getDeviceProperties(GPU_DEVICE)
    gpu_name = gpu_properties["name"]
    if isinstance(gpu_name, bytes):
        gpu_name = gpu_name.decode("utf-8")

    # Small allocation verifies that CuPy can actually execute on the device.
    gpu_test = cp.arange(10, dtype=cp.float32)
    _ = float(cp.sum(gpu_test).get())
    del gpu_test

    print(f"GPU training enabled: device {GPU_DEVICE} ({gpu_name})")
else:
    print("GPU training disabled.")

GPU training enabled: device 0 (NVIDIA GeForce RTX 3050 Laptop GPU)


## 3. Load and validate paired58 inputs

In [8]:
def load_feature_by_sample_matrix(path: Path, id_column: str, matrix_name: str,) -> pd.DataFrame:
    data = pd.read_csv(path, low_memory=False)

    if id_column not in data.columns:
        raise ValueError(f"{matrix_name} is missing identifier column '{id_column}'.")
    if data[id_column].isna().any() or (data[id_column].astype(str).str.strip() == "").any():
        raise ValueError(f"{matrix_name} contains missing or blank feature identifiers.")
    if data[id_column].duplicated().any():
        duplicated = data.loc[data[id_column].duplicated(), id_column].astype(str).unique()
        raise ValueError(f"{matrix_name} contains duplicated feature identifiers. "
                         f"Examples: {duplicated[:20].tolist()}")

    sample_columns = [column for column in data.columns if column != id_column]
    if not sample_columns:
        raise ValueError(f"{matrix_name} contains no sample columns.")

    try:
        matrix = data.loc[:, sample_columns].apply(pd.to_numeric, errors="raise")
    except Exception as exc:
        raise ValueError(f"{matrix_name} contains non-numeric sample values.") from exc

    matrix.index = data[id_column].astype(str)
    matrix.index.name = id_column
    matrix = matrix.astype(float)

    observed_values = matrix.to_numpy()[~np.isnan(matrix.to_numpy())]
    if observed_values.size == 0 or not np.isfinite(observed_values).all():
        raise ValueError(f"{matrix_name} contains no valid finite observations.")
    if matrix.isna().all(axis=1).any():
        raise ValueError(f"{matrix_name} contains features missing in all 58 samples.")
    if matrix.isna().all(axis=0).any():
        raise ValueError(f"{matrix_name} contains samples missing the entire view.")

    return matrix


metadata = pd.read_csv(METADATA_FILE, low_memory=False)
required_metadata_columns = {"SampleID", "Condition", "Paired_RNA_ProteinQC"}
missing_metadata_columns = required_metadata_columns - set(metadata.columns)
if missing_metadata_columns:
    raise ValueError("metadata_master_samples.csv is missing required columns: " + ", ".join(sorted(missing_metadata_columns)))

metadata["SampleID"] = metadata["SampleID"].astype(str)
if metadata["SampleID"].duplicated().any():
    raise ValueError("metadata_master_samples.csv contains duplicated SampleID values.")

if not pd.api.types.is_bool_dtype(metadata["Paired_RNA_ProteinQC"]):
    raise TypeError("Paired_RNA_ProteinQC must be a logical TRUE/FALSE column.")

paired_metadata = metadata.loc[metadata["Paired_RNA_ProteinQC"]].copy()
paired_ids = paired_metadata["SampleID"].tolist()

if len(paired_ids) != 58:
    raise ValueError(f"Expected 58 paired samples, but metadata contains {len(paired_ids)}.")

rna = load_feature_by_sample_matrix(RNA_FILE, "Gene", "RNA matrix")
protein = load_feature_by_sample_matrix(PROTEIN_FILE, "ProteinRowID", "Protein matrix",)
phosphopeptide = load_feature_by_sample_matrix(PHOSPHOPEPTIDE_FILE, "PeptideRowID", "Phosphopeptide matrix",)

for view_name, matrix in {"RNA": rna, "Protein": protein, "Phosphopeptide": phosphopeptide,}.items():
    missing = sorted(set(paired_ids) - set(matrix.columns))
    extra = sorted(set(matrix.columns) - set(paired_ids))
    if missing or extra:
        raise ValueError(f"{view_name} sample IDs do not match the paired58 metadata. "
                         f"Missing: {missing}; extra: {extra}")

rna = rna.loc[:, paired_ids]
protein = protein.loc[:, paired_ids]
phosphopeptide = phosphopeptide.loc[:, paired_ids]

if rna.isna().any().any():
    raise ValueError("The paired58 RNA input unexpectedly contains missing values.")

print("RNA:", rna.shape, "features x samples")
print("Protein:", protein.shape, "features x samples")
print("Phosphopeptide:", phosphopeptide.shape, "features x samples")
print("Condition composition:")
print(paired_metadata["Condition"].value_counts().rename_axis("Condition").to_frame("n").to_string())

RNA: (2000, 58) features x samples
Protein: (2000, 58) features x samples
Phosphopeptide: (181, 58) features x samples
Condition composition:


,n
Condition,
Treated,31
Control,27


## 4. Convert matrices to MOFA long format

View prefixes make feature names unique across modalities, following the suffix
logic used in the first-manuscript training notebook.

In [10]:
def matrix_to_mofa_long(matrix: pd.DataFrame, view_name: str) -> pd.DataFrame:
    output = matrix.copy()
    output.index = [f"{view_name}::{feature}" for feature in output.index.astype(str)]
    output.index.name = "feature"

    output = (output.reset_index().melt(id_vars="feature", var_name="sample", value_name="value"))
    output["view"] = view_name
    output["group"] = "C10_single_group"
    return output[["sample", "feature", "view", "group", "value"]]


rna_long = matrix_to_mofa_long(rna, "RNA")
protein_long = matrix_to_mofa_long(protein, "Protein")
phosphopeptide_long = matrix_to_mofa_long(phosphopeptide, "Phosphopeptide")

mofa_long = pd.concat([rna_long, protein_long, phosphopeptide_long], ignore_index=True,)

if mofa_long.duplicated(["sample", "feature", "view", "group"]).any():
    raise ValueError("Duplicated sample-feature-view entries were created.")

print("MOFA long-format dimensions:", mofa_long.shape)
print(mofa_long.groupby("view", sort=False).agg(Samples=("sample", "nunique"), Features=("feature", "nunique"),
                                                ObservedValues=("value", "count"),).to_string())

MOFA long-format dimensions: (242498, 5)


,Samples,Features,ObservedValues
view,,,
RNA,58,2000,116000
Protein,58,2000,106527
Phosphopeptide,58,181,6941


## 5. Train the native paired58 MOFA+ model

This intentionally mirrors the core settings of the first manuscript:
five factors, slow convergence, `dropR2=0.001`, GPU training and `seed=1`.

In [ ]:
ent = entry_point()

# Inputs are already feature-wise z-scored, so no additional view scaling or group centering is required.
ent.set_data_options(scale_views=False, scale_groups=False, center_groups=False, use_float32=True, )

ent.set_data_df(mofa_long, likelihoods=["gaussian", "gaussian", "gaussian"], )

ent.set_model_options(factors=N_FACTORS)

ent.set_train_options(convergence_mode="slow", dropR2=0.001, gpu_mode=GPU_MODE, seed=SEED, )

ent.build()
ent.run()
ent.save(str(MODEL_FILE), save_data=True)


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
       
 
        
use_float32 set to True: replacing float64 arrays by float32 arrays to speed up computations...



Loaded group='C10_single_group' view='Phosphopeptide' with N=58 samples and D=181 features...
Loaded group='C10_single_group' view='Protein' with N=58 samples and D=2000 features...
Loaded group='C10_single_group' view='RNA' with N=58 samples and D=2000 features...


Model options:

## 6. Save metadata and report completion

In [ ]:
paired_metadata.to_csv(
    OUTPUT_DIR / "native_paired58_metadata.csv",
    index=False,
)

print("\nNative paired58 MOFA+ training complete.")
print("Model:", MODEL_FILE)
print("Metadata:", OUTPUT_DIR / "native_paired58_metadata.csv")